# K-Means Project Tutorial: House grouping system

Classify California census-block groups by **region** and **median income** using `MedInc`, `Latitude`, and `Longitude`.

The train/test split is not used for supervised metrics here. We fit K-Means on `train`, then assign each unseen `test` house to the cluster it belongs to.

## Step 1: Loading the dataset

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/housing.csv")

total_data = pd.read_csv(DATA_PATH)
print(f"Shape: {total_data.shape}")
print(f"Missing values: {total_data.isnull().sum().sum()}")
total_data.head()

### Keep only the clustering features

In [ ]:
features = ["MedInc", "Latitude", "Longitude"]
X = total_data[features].copy()

print(X.describe())
X.head()

### Split into train and test

K-Means will learn the cluster centers from `X_train`. `X_test` is held out so we can later predict which cluster new houses belong to.

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
X_train.head()

## Step 2: Build a K-Means

Fit 6 clusters on the training houses, then store the assigned group as `cluster`.

In [ ]:
from sklearn.cluster import KMeans

model_unsup = KMeans(n_clusters=6, n_init="auto", random_state=42)
model_unsup.fit(X_train[features])

X_train["cluster"] = model_unsup.labels_
X_train["cluster"] = X_train["cluster"].astype("category")

print(X_train["cluster"].value_counts().sort_index())
X_train.head()

### Plot the training clusters

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axis = plt.subplots(1, 3, figsize=(15, 5))

sns.scatterplot(ax=axis[0], data=X_train, x="Longitude", y="Latitude", hue="cluster", palette="deep")
sns.scatterplot(ax=axis[1], data=X_train, x="Latitude", y="MedInc", hue="cluster", palette="deep")
sns.scatterplot(ax=axis[2], data=X_train, x="Longitude", y="MedInc", hue="cluster", palette="deep")

axis[0].set_title("Geographic clusters")
axis[1].set_title("Income vs latitude")
axis[2].set_title("Income vs longitude")
plt.tight_layout()
plt.show()

The map plot separates California into regional groups (Bay Area, Central Valley, Southern California, and inland areas). The income plots show that a few clusters capture higher-income coastal pockets, while others group lower-income inland blocks.

## Step 3: Predict clusters for new points

Use the trained K-Means model to assign each test house to the nearest cluster center.

In [ ]:
X_test["cluster"] = model_unsup.predict(X_test[features])
X_test["cluster"] = X_test["cluster"].astype("category")

print(X_test["cluster"].value_counts().sort_index())
X_test.head()

### Overlay test predictions on the training plot

In [ ]:
fig, axis = plt.subplots(1, 3, figsize=(15, 5))

sns.scatterplot(ax=axis[0], data=X_train, x="Longitude", y="Latitude", hue="cluster", palette="deep", alpha=0.15)
sns.scatterplot(ax=axis[1], data=X_train, x="Latitude", y="MedInc", hue="cluster", palette="deep", alpha=0.15)
sns.scatterplot(ax=axis[2], data=X_train, x="Longitude", y="MedInc", hue="cluster", palette="deep", alpha=0.15)

sns.scatterplot(ax=axis[0], data=X_test, x="Longitude", y="Latitude", hue="cluster", palette="deep", marker="+", legend=False)
sns.scatterplot(ax=axis[1], data=X_test, x="Latitude", y="MedInc", hue="cluster", palette="deep", marker="+", legend=False)
sns.scatterplot(ax=axis[2], data=X_test, x="Longitude", y="MedInc", hue="cluster", palette="deep", marker="+", legend=False)

axis[0].set_title("Test points over training clusters")
axis[1].set_title("Income vs latitude")
axis[2].set_title("Income vs longitude")
plt.tight_layout()
plt.show()

The `+` markers (test houses) land inside the same geographic and income groups as the faded training points. That means the model is assigning new houses to the clusters learned from `train`.

In [ ]:
from pathlib import Path
from pickle import dump

processed_dir = Path("../data/processed")
models_dir = Path("../models")
processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "housing_train.csv", index=False)
X_test.to_csv(processed_dir / "housing_test.csv", index=False)

with open(models_dir / "kmeans_housing.sav", "wb") as file:
    dump(model_unsup, file)

print("Saved train/test CSVs and the K-Means model.")